# Audio Classification: MotoSense

Dataset processing and CNN/ViT pipeline for motorcycle engine classification.

Pipeline:
1. Data renaming
2. Train/val/test split
3. Preprocessing (Trim & Normalize) & Chunking (Sliding Window)
4. Augmentation (Training set only)
5. Log-Mel Spectrogram Extraction
6. Model Training & Evaluation

## 1. Library & Konfigurasi

In [19]:
import os, glob, shutil, random, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import librosa
import soundfile as sf

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import Sequential, layers, regularizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from audiomentations import (
    Compose, AddGaussianNoise, PitchShift,
    TimeStretch, Shift, PolarityInversion
)

import matplotlib.pyplot as plt
import seaborn as sns

# ── Konfigurasi ──────────────────────────────────────────────
CLASSES       = ["Clutch-Shoe", "Conecting-Rod", "Drive-Belt",
                 "Piston", "Tensioner", "Slider", "Roller", "Face-Drive"]
BASE_DIR      = "Dataset"
TARGET_SR     = 16000
DURATION      = 3
TARGET_SAMPLES = TARGET_SR * DURATION  # 48.000
STRIDE_SAMPLES = int(TARGET_SR * 1.5)

# Mel-Spectrogram — shape output: (128, 128)
N_MELS       = 128
N_FFT        = 1024
HOP_LENGTH   = 375  # 48000 / 375 = 128 frame tepat

# Augmentasi
AUG_TARGET   = 500  # minimal per kelas di training

# Training
BATCH_SIZE   = 16
EPOCHS       = 100
SEED         = 42

# ── Folder paths ─────────────────────────────────────────────
PATHS = {
    "raw"         : os.path.join(BASE_DIR, "part-rusak"),
    "rename"      : os.path.join(BASE_DIR, "rename"),
    "split_orig"  : os.path.join(BASE_DIR, "split_original"),
    "prep_train"  : os.path.join(BASE_DIR, "preprocesed", "train"),
    "prep_val"    : os.path.join(BASE_DIR, "preprocesed", "val"),
    "prep_test"   : os.path.join(BASE_DIR, "preprocesed", "test"),
    "aug_train"   : os.path.join(BASE_DIR, "augmented", "train"),
    "features"    : os.path.join(BASE_DIR, "features"),
    "models"      : "models",
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU         : {tf.config.list_physical_devices('GPU') or 'CPU'}")
print(f"Target shape: ({N_MELS}, {TARGET_SAMPLES // HOP_LENGTH}, 1)")

TensorFlow  : 2.21.0
GPU         : CPU
Target shape: (128, 128, 1)


## 2. Rename File

In [20]:
for cls in CLASSES:
    src = os.path.join(PATHS['raw'], cls)
    dst = os.path.join(PATHS['rename'], cls)
    os.makedirs(dst, exist_ok=True)
    files = sorted(f for f in os.listdir(src) if f.endswith(('.wav','.mp3','.m4a')))
    for i, f in enumerate(files, 1):
        ext = os.path.splitext(f)[1]
        shutil.copy(os.path.join(src, f),
                    os.path.join(dst, f"{cls.lower()}-{i:03d}{ext}"))
    print(f"  {cls:<15}: {len(files)} file")

print(f"\nDisimpan ke: {PATHS['rename']}")

  Clutch-Shoe    : 15 file
  Conecting-Rod  : 10 file
  Drive-Belt     : 20 file
  Piston         : 6 file
  Tensioner      : 8 file
  Slider         : 13 file
  Roller         : 19 file
  Face-Drive     : 9 file

Disimpan ke: Dataset/rename


## 3. Train/Val/Test Split

Split data awal untuk mencegah data leakage.

In [21]:
all_files, all_labels = [], []
for lbl, cls in enumerate(CLASSES):
    for f in glob.glob(os.path.join(PATHS['rename'], cls, '*.wav')):
        all_files.append(f)
        all_labels.append(lbl)

# Split: 70 / 15 / 15
X_tmp, X_test, y_tmp, y_test = train_test_split(
    all_files, all_labels, test_size=0.15, stratify=all_labels, random_state=SEED)
X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    X_tmp, y_tmp, test_size=0.176, stratify=y_tmp, random_state=SEED)

# Menyalin file asli ke masing-masing folder split
for name, files, labels in [("train", X_train, y_train_raw),
                              ("val",   X_val,   y_val_raw),
                              ("test",  X_test,  y_test)]:
    count = 0
    for f, l in zip(files, labels):
        cls = CLASSES[l]
        dst_dir = os.path.join(PATHS['split_orig'], name, cls)
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(f, os.path.join(dst_dir, os.path.basename(f)))
        count += 1
    print(f"  {name:<6}: {count:>3} file disalin ke {os.path.join(PATHS['split_orig'], name)}")

print(f"\nTotal file asli: {len(all_files)}")

  train :  70 file disalin ke Dataset/split_original/train
  val   :  15 file disalin ke Dataset/split_original/val
  test  :  15 file disalin ke Dataset/split_original/test

Total file asli: 100


## 4. Preprocessing & Chunking

Sliding window chunking (3 detik) untuk tiap subset.

In [22]:
def preprocess(path):
    y, _ = librosa.load(path, sr=TARGET_SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    peak  = np.max(np.abs(y))
    return y / peak if peak > 0 else y

def chunk_and_save(src_base, out_base):
    total = 0
    for cls in CLASSES:
        src_dir = os.path.join(src_base, cls)
        dst_dir = os.path.join(out_base, cls)
        os.makedirs(dst_dir, exist_ok=True)
        
        files = glob.glob(os.path.join(src_dir, '*.wav'))
        for f in files:
            audio  = preprocess(f)
            base   = os.path.splitext(os.path.basename(f))[0]

            # Sliding window
            audio_len = len(audio)
            if audio_len < TARGET_SAMPLES:
                audio = np.tile(audio, int(np.ceil(TARGET_SAMPLES / audio_len)))
            chunks = [audio[i:i+TARGET_SAMPLES]
                      for i in range(0, len(audio)-TARGET_SAMPLES+1, STRIDE_SAMPLES)
                      if len(audio[i:i+TARGET_SAMPLES]) == TARGET_SAMPLES]
            for ci, chunk in enumerate(chunks):
                sf.write(os.path.join(dst_dir, f"{base}_c{ci}.wav"), chunk, TARGET_SR, subtype='PCM_16')
                total += 1
    return total

for subset, out_key in [("train", "prep_train"), ("val", "prep_val"), ("test", "prep_test")]:
    src_base = os.path.join(PATHS['split_orig'], subset)
    n = chunk_and_save(src_base, PATHS[out_key])
    print(f"  {subset:<6}: {n} chunks → {PATHS[out_key]}")

print("\nChunking selesai")

  train : 107 chunks → Dataset/preprocesed/train
  val   : 20 chunks → Dataset/preprocesed/val
  test  : 20 chunks → Dataset/preprocesed/test

Chunking selesai


## 5. Augmentation

Hanya diterapkan pada data training.

In [23]:
augment = Compose([
    AddGaussianNoise(min_amplitude=0.002, max_amplitude=0.020, p=0.6),
    PitchShift(min_semitones=-3, max_semitones=3, p=0.6),
    TimeStretch(min_rate=0.85, max_rate=1.15, p=0.5),
    Shift(min_shift=-0.2, max_shift=0.2, p=0.4),
    PolarityInversion(p=0.3),
])

for cls in CLASSES:
    src_dir = os.path.join(PATHS['prep_train'], cls)
    dst_dir = os.path.join(PATHS['aug_train'],  cls)
    os.makedirs(dst_dir, exist_ok=True)

    existing = glob.glob(os.path.join(src_dir, '*.wav'))
    # Salin file asli
    for f in existing:
        shutil.copy(f, os.path.join(dst_dir, os.path.basename(f)))

    needed = max(0, AUG_TARGET - len(existing))
    for i in range(needed):
        y, _ = librosa.load(random.choice(existing), sr=TARGET_SR, mono=True)
        y_aug = augment(samples=y.astype(np.float32), sample_rate=TARGET_SR)
        if len(y_aug) < TARGET_SAMPLES:
            y_aug = np.pad(y_aug, (0, TARGET_SAMPLES - len(y_aug)))
        sf.write(os.path.join(dst_dir, f"aug_{i:05d}.wav"),
                 y_aug[:TARGET_SAMPLES], TARGET_SR, subtype='PCM_16')

    final = len(glob.glob(os.path.join(dst_dir, '*.wav')))
    print(f"  {cls:<15}: {len(existing):>3} asli → {final} total (+{needed} aug)")

print(f"\nDisimpan ke: {PATHS['aug_train']}")

  Clutch-Shoe    :  15 asli → 500 total (+485 aug)
  Conecting-Rod  :   8 asli → 500 total (+492 aug)
  Drive-Belt     :  22 asli → 500 total (+478 aug)
  Piston         :   4 asli → 500 total (+496 aug)
  Tensioner      :  11 asli → 500 total (+489 aug)
  Slider         :  17 asli → 500 total (+483 aug)
  Roller         :  22 asli → 500 total (+478 aug)
  Face-Drive     :   8 asli → 500 total (+492 aug)

Disimpan ke: Dataset/augmented/train


## 6. Feature Extraction (Log-Mel Spectrogram)

Konversi audio ke matriks 2D (128x128).

### Visualisasi Spectrogram

Sample perbandingan waveform dan spectrogram.

In [24]:
# # Ambil satu contoh file audio dari hasil augmentasi (atau prep_train jika tidak ada aug_train)
# sample_files = glob.glob(os.path.join(PATHS['aug_train'], CLASSES[0], '*.wav'))
# if not sample_files:
#     sample_files = glob.glob(os.path.join(PATHS['prep_train'], CLASSES[0], '*.wav'))
# sample_file = sample_files[0]

# # Baca audio
# y, _ = librosa.load(sample_file, sr=TARGET_SR, mono=True)
# if len(y) < TARGET_SAMPLES:
#     y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
# y = y[:TARGET_SAMPLES]

# # Hitung melspectrogram
# mel = librosa.feature.melspectrogram(
#     y=y, sr=TARGET_SR, n_fft=N_FFT,
#     hop_length=HOP_LENGTH, n_mels=N_MELS
# )
# log_mel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)

# # Plotting
# plt.figure(figsize=(12, 4))

# plt.subplot(1, 2, 1)
# librosa.display.waveshow(y, sr=TARGET_SR)
# plt.title("Waveform Audio")

# plt.subplot(1, 2, 2)
# librosa.display.specshow(log_mel, sr=TARGET_SR, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel', cmap='viridis')
# plt.colorbar(format='%+2.0f dB')
# plt.title(" Citra Spectrogram")

# plt.tight_layout()
# plt.show()

# print(f"Bentuk (shape) matriks fitur citra: {log_mel.shape}")


In [25]:
# def extract_log_mel(wav_path):
#     """Load .wav, ekstrak log-mel spectrogram shape (N_MELS, T)."""
#     y, _ = librosa.load(wav_path, sr=TARGET_SR, mono=True)
#     if len(y) < TARGET_SAMPLES:
#         y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
#     y = y[:TARGET_SAMPLES]
#     mel = librosa.feature.melspectrogram(
#         y=y, sr=TARGET_SR, n_fft=N_FFT,
#         hop_length=HOP_LENGTH, n_mels=N_MELS
#     )
#     return librosa.power_to_db(mel, ref=np.max).astype(np.float32)

# def build_features(data_dir):
#     X, y = [], []
#     for lbl, cls in enumerate(CLASSES):
#         files = glob.glob(os.path.join(data_dir, cls, '*.wav'))
#         for f in files:
#             X.append(extract_log_mel(f))
#             y.append(lbl)
#     return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

# FEAT = {
#     "train": (os.path.join(PATHS['features'], 'X_train.npy'),
#               os.path.join(PATHS['features'], 'y_train.npy')),
#     "val"  : (os.path.join(PATHS['features'], 'X_val.npy'),
#               os.path.join(PATHS['features'], 'y_val.npy')),
#     "test" : (os.path.join(PATHS['features'], 'X_test.npy'),
#               os.path.join(PATHS['features'], 'y_test.npy')),
# }

# SRC = {"train": PATHS['aug_train'], "val": PATHS['prep_val'], "test": PATHS['prep_test']}

# for subset, (xp, yp) in FEAT.items():
#     if os.path.exists(xp):
#         print(f"  {subset:<6}: cache ditemukan, skip")
#         continue
#     X, y = build_features(SRC[subset])
#     np.save(xp, X); np.save(yp, y)
#     print(f"  {subset:<6}: {X.shape}  → disimpan")

# print(f"\nSemua fitur tersimpan di: {PATHS['features']}")

## 7. Load Features

In [26]:
# X_train = np.load(FEAT['train'][0])
# y_train = np.load(FEAT['train'][1])
# X_val   = np.load(FEAT['val'][0])
# y_val   = np.load(FEAT['val'][1])
# X_test  = np.load(FEAT['test'][0])
# y_test  = np.load(FEAT['test'][1])

# # Tambah channel dim: (N, 128, 128) → (N, 128, 128, 1)
# X_train = X_train[..., np.newaxis]
# X_val   = X_val[...,   np.newaxis]
# X_test  = X_test[...,  np.newaxis]

# # Normalisasi per-sampel ke [0, 1]
# def norm(X):
#     mn = X.min(axis=(1,2,3), keepdims=True)
#     mx = X.max(axis=(1,2,3), keepdims=True)
#     return (X - mn) / (mx - mn + 1e-8)

# X_train, X_val, X_test = norm(X_train), norm(X_val), norm(X_test)

# print(f"X_train : {X_train.shape}  y_train : {y_train.shape}")
# print(f"X_val   : {X_val.shape}  y_val   : {y_val.shape}")
# print(f"X_test  : {X_test.shape}  y_test  : {y_test.shape}")

## 8. Arsitektur Model — CNN

CNN dipilih karena memperlakukan log-mel spectrogram seperti gambar:
- **Sumbu Y**: frekuensi (mel bands)
- **Sumbu X**: waktu (frames)
- Conv2D belajar pola frekuensi × waktu yang khas per jenis kerusakan

`GlobalAveragePooling2D` menggantikan Flatten untuk mengurangi parameter dan overfitting.

In [27]:
# INPUT_SHAPE = X_train.shape[1:]  # (128, 128, 1)
# N_CLASSES   = len(CLASSES)

# def build_cnn(input_shape, n_classes):
#     return Sequential([
#         layers.Input(shape=input_shape),

#         # Block 1
#         layers.Conv2D(16, (3,3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
#         layers.BatchNormalization(),
#         layers.Conv2D(16, (3,3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
#         layers.MaxPooling2D((2,2)),
#         layers.Dropout(0.30),

#         # Block 2
#         layers.Conv2D(32, (3,3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
#         layers.BatchNormalization(),
#         layers.Conv2D(32, (3,3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
#         layers.MaxPooling2D((2,2)),
#         layers.Dropout(0.30),

#         # Block 3
#         layers.Conv2D(64, (3,3), padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
#         layers.BatchNormalization(),
#         layers.MaxPooling2D((2,2)),
#         layers.Dropout(0.40),

#         # Classifier head
#         layers.GlobalAveragePooling2D(),
#         layers.Dense(128, activation='relu',
#                      kernel_regularizer=regularizers.l2(1e-3)),
#         layers.Dropout(0.6),
#         layers.Dense(n_classes, activation='softmax'),
#     ], 
#     name='motosense_cnn')

# model = build_cnn(INPUT_SHAPE, N_CLASSES)
# model.summary()
# print(f"\nTotal parameter: {model.count_params():,}")

## Alternatif: Vision Transformer (ViT)

Compact ViT untuk perbandingan performa.

In [28]:
# import tensorflow as tf
# from tensorflow.keras import layers, Model

# def build_transformer(input_shape, n_classes, patch_size=16, embed_dim=32, num_heads=4, ff_dim=128, num_transformer_blocks=3):
#     """
#     Membangun arsitektur Compact Vision Transformer (ViT) untuk Spectrogram.
#     """
#     inputs = layers.Input(shape=input_shape)
    
#     # 1. Patch Extraction & Linear Projection
#     # Memotong citra 128x128 menjadi kotak-kotak berukuran 16x16 (total 64 patch).
#     patches = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding="VALID", name="patch_extract")(inputs)
    
#     # Reshape menjadi urutan (sequence) patch: (Batch, 64, embed_dim)
#     seq_len = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)
#     x = layers.Reshape((seq_len, embed_dim))(patches)
    
#     # 2. Positional Embedding
#     # Transformer tidak mengerti urutan, jadi kita harus menambahkan informasi posisi untuk tiap patch
#     positions = tf.range(start=0, limit=seq_len, delta=1)
#     position_embedding = layers.Embedding(input_dim=seq_len, output_dim=embed_dim, name="pos_embedding")(positions)
#     x = x + position_embedding
    
#     # 3. Transformer Encoder Blocks
#     for _ in range(num_transformer_blocks):
#         # Layer Normalization 1
#         x1 = layers.LayerNormalization(epsilon=1e-6)(x)
#         # Multi-Head Self Attention
#         attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=0.3)(x1, x1)
#         # Skip connection 1
#         x2 = layers.Add()([attention_output, x])
        
#         # Layer Normalization 2
#         x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
#         # MLP (Multi-Layer Perceptron)
#         x3 = layers.Dense(ff_dim, activation=tf.nn.gelu)(x3)
#         x3 = layers.Dropout(0.1)(x3)
#         x3 = layers.Dense(embed_dim)(x3)
#         x3 = layers.Dropout(0.1)(x3)
#         # Skip connection 2
#         x = layers.Add()([x3, x2])
        
#     # 4. Classification Head
#     x = layers.LayerNormalization(epsilon=1e-6)(x)
#     x = layers.GlobalAveragePooling1D()(x) # Menggabungkan informasi dari semua patch
#     x = layers.Dropout(0.5)(x)
#     outputs = layers.Dense(n_classes, activation="softmax")(x)
    
#     return Model(inputs=inputs, outputs=outputs, name="motosense_transformer")

# # model_transformer = build_transformer(INPUT_SHAPE, N_CLASSES)
# # model_transformer.summary()

## 9. Training

In [29]:
# cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
# cw_dict = dict(enumerate(cw))

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# os.makedirs(PATHS['models'], exist_ok=True)
# MODEL_PATH = os.path.join(PATHS['models'], 'best_model.keras')

# callbacks = [
#     ModelCheckpoint(MODEL_PATH, monitor='val_accuracy',
#                     save_best_only=True, verbose=0),
#     EarlyStopping(monitor='val_loss', patience=20,
#                   restore_best_weights=True, verbose=1),
#     ReduceLROnPlateau(monitor='val_loss', factor=0.5,
#                       patience=5, min_lr=1e-6, verbose=1),
# ]

# history = model.fit(
#     X_train, y_train,
#     validation_data=(X_val, y_val),
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     class_weight=cw_dict,
#     callbacks=callbacks,
#     verbose=1
# )

# best_val = max(history.history['val_accuracy'])
# best_ep  = int(np.argmax(history.history['val_accuracy'])) + 1
# print(f"\nBest val_accuracy: {best_val:.4f} (epoch {best_ep})")
# print(f"Model disimpan   : {MODEL_PATH}")

## 10. Evaluasi

In [30]:
# # Learning curve
# fig, ax = plt.subplots(1, 2, figsize=(12, 4))
# ep = range(1, len(history.history['loss']) + 1)
# ax[0].plot(ep, history.history['accuracy'],     label='train')
# ax[0].plot(ep, history.history['val_accuracy'], label='val')
# ax[0].set_title('Accuracy'); ax[0].legend(); ax[0].grid(alpha=0.3)
# ax[1].plot(ep, history.history['loss'],     label='train')
# ax[1].plot(ep, history.history['val_loss'], label='val')
# ax[1].set_title('Loss'); ax[1].legend(); ax[1].grid(alpha=0.3)
# plt.tight_layout()
# # plt.savefig(os.path.join(PATHS['models'], 'learning_curve.png'), dpi=120)
# plt.show()

In [31]:
# # Evaluasi pada test set
# best_model = tf.keras.models.load_model(MODEL_PATH)
# y_pred = best_model.predict(X_test, verbose=0).argmax(axis=1)

# print(classification_report(y_test, y_pred, target_names=CLASSES, digits=3))

# # Confusion matrix
# cm = confusion_matrix(y_test, y_pred)
# plt.figure(figsize=(9, 7))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=CLASSES, yticklabels=CLASSES)
# plt.xlabel('Predicted'); plt.ylabel('True')
# plt.title('Confusion Matrix — Test Set')
# plt.xticks(rotation=45, ha='right')
# plt.tight_layout()
# plt.savefig(os.path.join(PATHS['models'], 'confusion_matrix.png'), dpi=120)
# plt.show()

In [32]:
# def predict_file(wav_path, model, threshold=0.50):
#     y, _ = librosa.load(wav_path, sr=TARGET_SR, mono=True)
#     if len(y) < TARGET_SAMPLES:
#         y = np.pad(y, (0, TARGET_SAMPLES - len(y)))
#     y = y[:TARGET_SAMPLES]
#     mel = librosa.feature.melspectrogram(
#         y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
#     feat = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
#     feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
#     feat = feat[np.newaxis, ..., np.newaxis]  # (1, 128, 128, 1)

#     probs = model.predict(feat, verbose=0)[0]
#     idx   = np.argmax(probs)
#     return {
#         "label"     : CLASSES[idx] if probs[idx] >= threshold else "tidak_yakin",
#         "confidence": f"{probs[idx]*100:.1f}%",
#         "all"       : {c: f"{p*100:.1f}%" for c,p in zip(CLASSES, probs)}
#     }

# # Contoh pemakaian
# # result = predict_file("path/ke/audio.wav", best_model)
# # print(result)

## Ekstra: Eksperimen Transfer Learning dengan YAMNet

Sel ini adalah area *Playground* mandiri. Kode ini **tidak merusak atau mengubah** model CNN/ViT dan pipeline ekstraksi `.npy` yang sudah Anda miliki di atas.
Ini bertujuan agar Anda dapat membandingkan akurasi jika Anda menggunakan model raksasa pra-latih (Pre-trained) dari Google (YAMNet).

Alur:
1. Mengambil bobot YAMNet dari *TensorFlow Hub*.
2. Membaca rekaman `.wav` yang sama persis dengan CNN.
3. YAMNet menerjemahkan audio menjadi vektor numerik (`1024-dim`). Vektor ini disimpan agar proses *training* ke depan tidak perlu mengekstrak audio ulang.
4. Melatih (*Transfer Learning*) _Dense layer_ sederhana untuk mengklasifikasikan 7 jenis mesin motor kita.

In [33]:
!pip install tensorflow_hub

import tensorflow_hub as hub
import numpy as np
import os
import glob
import librosa
import tensorflow as tf
from tensorflow.keras import layers, Sequential, regularizers
from sklearn.utils.class_weight import compute_class_weight

print("Memuat model YAMNet dari TF Hub...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

def extract_yamnet_embeddings(data_dir):
    """Ekstrak embedding 1024-dim dari YAMNet untuk setiap file audio"""
    X, y = [], []
    for lbl, cls in enumerate(CLASSES):
        files = glob.glob(os.path.join(data_dir, cls, '*.wav'))
        for f in files:
            wav, _ = librosa.load(f, sr=16000, mono=True)
            # Normalisasi waveform ke rentang [-1.0, 1.0] yang disukai YAMNet
            wav = wav / (np.max(np.abs(wav)) + 1e-8)
            # YAMNet inference
            _, embeddings, _ = yamnet_model(wav)
            # Ambil rata-rata pooling pada dimensi waktu (frames)
            emb_mean = np.mean(embeddings.numpy(), axis=0)
            X.append(emb_mean)
            y.append(lbl)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

YAMNET_FEAT = {
    "train": (os.path.join(PATHS['features'], 'X_train_yamnet.npy'), os.path.join(PATHS['features'], 'y_train_yamnet.npy')),
    "val"  : (os.path.join(PATHS['features'], 'X_val_yamnet.npy'),   os.path.join(PATHS['features'], 'y_val_yamnet.npy')),
    "test" : (os.path.join(PATHS['features'], 'X_test_yamnet.npy'),  os.path.join(PATHS['features'], 'y_test_yamnet.npy')),
}

# 1. Ekstrak & Simpan Fitur YAMNet
SRC = {"train": PATHS['aug_train'], "val": PATHS['prep_val'], "test": PATHS['prep_test']}

print("\nEkstraksi fitur YAMNet...")
for subset, (xp, yp) in YAMNET_FEAT.items():
    if os.path.exists(xp):
        print(f"  {subset:<6}: cache ditemukan, skip")
    else:
        print(f"  {subset:<6}: mengekstrak dari audio (ini membutuhkan sedikit waktu)...")
        X, y = extract_yamnet_embeddings(SRC[subset])
        np.save(xp, X); np.save(yp, y)
        print(f"  {subset:<6}: {X.shape} tersimpan")

# 2. Load Fitur
X_train_yn, y_train_yn = np.load(YAMNET_FEAT['train'][0]), np.load(YAMNET_FEAT['train'][1])
X_val_yn, y_val_yn     = np.load(YAMNET_FEAT['val'][0]),   np.load(YAMNET_FEAT['val'][1])
X_test_yn, y_test_yn   = np.load(YAMNET_FEAT['test'][0]),  np.load(YAMNET_FEAT['test'][1])

# 3. Bangun Arsitektur Classifier YAMNet
def build_yamnet_classifier(n_classes):
    return Sequential([
        layers.Input(shape=(1024,)),
        layers.Dropout(0.4),
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.4),
        layers.Dense(n_classes, activation='softmax')
    ], name="yamnet_classifier")

model_yn = build_yamnet_classifier(len(CLASSES))
model_yn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 4. Training (Transfer Learning)
print("\nTraining YAMNet Classifier...")
cw_yn = compute_class_weight('balanced', classes=np.unique(y_train_yn), y=y_train_yn)
history_yn = model_yn.fit(
    X_train_yn, y_train_yn,
    validation_data=(X_val_yn, y_val_yn),
    epochs=40,
    batch_size=32,
    class_weight=dict(enumerate(cw_yn)),
    verbose=1
)

# 5. Evaluasi Cepat
loss, acc = model_yn.evaluate(X_test_yn, y_test_yn, verbose=0)
print(f"\n---> Akurasi YAMNet pada Test Set: {acc*100:.2f}%")

Memuat model YAMNet dari TF Hub...

Ekstraksi fitur YAMNet...
  train : mengekstrak dari audio (ini membutuhkan sedikit waktu)...
  train : (4000, 1024) tersimpan
  val   : mengekstrak dari audio (ini membutuhkan sedikit waktu)...
  val   : (20, 1024) tersimpan
  test  : mengekstrak dari audio (ini membutuhkan sedikit waktu)...
  test  : (20, 1024) tersimpan

Training YAMNet Classifier...
Epoch 1/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6357 - loss: 1.1106 - val_accuracy: 0.8000 - val_loss: 0.4967
Epoch 2/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7975 - loss: 0.6325 - val_accuracy: 0.8000 - val_loss: 0.4462
Epoch 3/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8372 - loss: 0.5023 - val_accuracy: 0.8000 - val_loss: 0.4486
Epoch 4/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8737 - loss: 0.4060 - val_accuracy: 0.8000 - val_loss: 0.5055
Epoch 5/40
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8808 - loss: 0.3725 - val_accu

## 11. Inference Pipeline

Pengujian model pada data raw `.wav`.

In [34]:
import random

def predict_single_audio(wav_path, model):
    """
    Memprediksi kelas dari satu file audio mentah (.wav).
    File dipotong-potong (chunking) seperti saat training,
    lalu hasil prediksi dari semua chunk akan di-voting (mayoritas).
    """
    # 1. Load & Preprocess
    y, _ = librosa.load(wav_path, sr=TARGET_SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    peak  = np.max(np.abs(y))
    y = y / peak if peak > 0 else y
    
    # 2. Chunking (Sliding Window)
    audio_len = len(y)
    if audio_len < TARGET_SAMPLES:
        y = np.tile(y, int(np.ceil(TARGET_SAMPLES / audio_len)))
        
    chunks = [y[i:i+TARGET_SAMPLES]
              for i in range(0, len(y)-TARGET_SAMPLES+1, STRIDE_SAMPLES)
              if len(y[i:i+TARGET_SAMPLES]) == TARGET_SAMPLES]
              
    if not chunks:
        chunks = [y[:TARGET_SAMPLES]]
        
    # 3. Ekstrak Fitur (Spectrogram)
    features = []
    for chunk in chunks:
        mel = librosa.feature.melspectrogram(
            y=chunk, sr=TARGET_SR, n_fft=N_FFT,
            hop_length=HOP_LENGTH, n_mels=N_MELS
        )
        db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
        features.append(db)
        
    X_infer = np.array(features)
    X_infer = X_infer[..., np.newaxis] # (N_chunks, 128, 128, 1)
    
    # 4. Normalisasi [0, 1]
    mn = X_infer.min(axis=(1,2,3), keepdims=True)
    mx = X_infer.max(axis=(1,2,3), keepdims=True)
    X_infer = (X_infer - mn) / (mx - mn + 1e-8)
    
    # 5. Prediksi
    preds = model.predict(X_infer, verbose=0)
    pred_classes = preds.argmax(axis=1)
    
    # Voting Mayoritas
    unique, counts = np.unique(pred_classes, return_counts=True)
    final_class_idx = unique[np.argmax(counts)]
    
    return CLASSES[final_class_idx]

# Mari kita uji coba dengan mengambil 5 file acak dari folder Data Test asli
print("--- Inference Test ---")

test_dir = os.path.join(PATHS['split_orig'], "test")
all_test_files = glob.glob(os.path.join(test_dir, "*", "*.wav"))

if all_test_files:
    # Memilih 5 file acak dari data test
    sample_files = random.sample(all_test_files, min(10, len(all_test_files)))
    
    for f in sample_files:
        true_label = os.path.basename(os.path.dirname(f))
        file_name = os.path.basename(f)
        
        # Prediksi menggunakan best_model
        predicted_label = predict_single_audio(f, best_model) 
        
        status = "OK" if true_label == predicted_label else "FAIL"
        print(f"[{status}] {file_name} | True: {true_label} | Pred: {predicted_label}")
else:
    print("Tidak ditemukan file audio di folder test. Pastikan dataset sudah diekstrak.")

--- Inference Test ---
[OK] drive-belt-019.wav | True: Drive-Belt | Pred: Drive-Belt
[FAIL] slider-005.wav | True: Slider | Pred: Tensioner
[FAIL] face-drive-001.wav | True: Face-Drive | Pred: Roller
[OK] drive-belt-005.wav | True: Drive-Belt | Pred: Drive-Belt
[FAIL] slider-002.wav | True: Slider | Pred: Tensioner
[OK] clutch-shoe-013.wav | True: Clutch-Shoe | Pred: Clutch-Shoe
[OK] tensioner-004.wav | True: Tensioner | Pred: Tensioner
[FAIL] clutch-shoe-010.wav | True: Clutch-Shoe | Pred: Tensioner
[FAIL] drive-belt-018.wav | True: Drive-Belt | Pred: Tensioner
[OK] roller-003.wav | True: Roller | Pred: Roller
